# Stage 1 — bbox + track ID overlay

Loads a clip + its `tracks_*.json` produced by `scripts/stage1_run.py` (or the Modal `run_yolo_bytetrack` function), draws bounding boxes and track IDs on each frame, and writes the result as `outputs/stage1_<session>_<cam>_overlay.mp4` so the qualitative ID-stability check (Phase B user test step §10) is one command.

Edit the two paths in the next cell.

In [ ]:
from pathlib import Path

import cv2
import numpy as np

from courtvision.io_utils import read_json_as
from courtvision.schemas import TrackSequence

VIDEO_PATH = Path('outputs/clips/sample.mp4')         # <- edit me
TRACKS_JSON = Path('outputs/stage1/ms_demo/tracks_ms_demo_cam0.json')  # <- edit me

In [ ]:
seq = read_json_as(TrackSequence, TRACKS_JSON)
print(f'{seq.session_id}/{seq.camera_id}: {len(seq.tracks)} tracks across {seq.n_frames} frames @ {seq.fps:.1f} fps')
by_frame = {}
for t in seq.tracks:
    by_frame.setdefault(t.frame_idx, []).append(t)

rng = np.random.default_rng(0)
# Stable per-id colour palette.
track_ids = sorted({t.track_id for t in seq.tracks})
colours = {tid: tuple(int(c) for c in rng.integers(60, 255, size=3)) for tid in track_ids}

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or seq.fps
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = Path('outputs') / f'stage1_{seq.session_id}_{seq.camera_id}_overlay.mp4'
out_path.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    for t in by_frame.get(frame_idx, []):
        x, y, bw, bh = t.bbox
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + bw), int(y + bh)
        c = colours[t.track_id]
        cv2.rectangle(frame, (x1, y1), (x2, y2), c, 2)
        cv2.putText(frame, f'id {t.track_id} ({t.confidence:.2f})', (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1, cv2.LINE_AA)
    writer.write(frame)
    frame_idx += 1

writer.release()
cap.release()
print(f'wrote {out_path} ({frame_idx} frames)')